# GEANT h96 Matrix3 recovery — only shuffled seeds 123 and 2026

A fresh, non-resumable recovery namespace. This notebook pins the Review18 source candidate, which requires independent review and Colab preflight before any training authorization. It validates the six original Drive backups read-only, verifies the source/data packages, runs synthetic tests and a no-training preflight. Training is disabled by default and requires source review, a global confirmation, and a distinct exact typed authorization for each remaining run. Each completed run is read-back hash-verified on Drive before the next authorization. No test split is included or opened.

The original Matrix3 Drive backups cannot be inspected from this local authoring session; their real verification occurs in Colab after Drive mount. Drive FUSE read-back is not a guarantee of cloud replication durability. Any partial recovery directory is preserved and must never be resumed.

In [ ]:
import sys, subprocess, importlib.metadata as ilm
if sys.version_info[:2] != (3, 12): raise RuntimeError(f'Requires Python 3.12; found {sys.version.split()[0]}')
PINS={'numpy':'1.26.4','torch':'2.12.0','psutil':'7.2.2','pytest':'9.1.1'}
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',*[f'{k}=={v}' for k,v in PINS.items()]],check=True)
actual={k:ilm.version(k) for k in PINS}
if actual!=PINS: raise RuntimeError(f'Pinned versions not active: {actual}; restart runtime and rerun from cell 1')
import torch
if torch.cuda.is_available(): raise RuntimeError('This protocol is CPU-only')
print('PASS: environment pins and CPU runtime',actual)

In [ ]:
import hashlib, json, os, stat, uuid, zipfile
from pathlib import Path, PurePosixPath
from google.colab import files, drive
drive.mount('/content/drive')
EXPECTED={'geant_h96_colab_source_v1.zip':'81aa2b765b72e4eab00822010414d2446dfa2e1b8164891cc232dbc146172f20','source_snapshot_manifest.json':'592715fa389d53624bdce2749267d669ed40c1d209369f0e1d255e2baf8dffee','geant_h96_colab_trainval_v1.zip':'df2bac044c9b9a577d5117310a99022cce402b9370e2ba483ea18b8d7045521e','dataset_bundle_manifest.json':'70f60f09b71c0cb89003ac2987af4433df0eeeb64b696845faa25d0fbadb8a95'}
def sha_bytes(b): return hashlib.sha256(b).hexdigest()
uploaded=files.upload()
if len(uploaded)!=4: raise RuntimeError(f'Upload exactly the four approved bundles/manifests; got {len(uploaded)}')
by_hash={}
for _name,b in uploaded.items(): by_hash.setdefault(sha_bytes(b),[]).append(b)
resolved={}
for name,digest in EXPECTED.items():
 matches=by_hash.get(digest,[])
 if len(matches)!=1: raise RuntimeError(f'Missing, duplicate, or mismatched upload: {name}')
 resolved[name]=matches[0]
if sum(map(len,by_hash.values()))!=4: raise RuntimeError('Unexpected or duplicate upload refused')
DRIVE_BASE=Path('/content/drive/MyDrive/SDNResearch')
if not DRIVE_BASE.is_dir() or DRIVE_BASE.resolve(strict=True)!=DRIVE_BASE: raise RuntimeError('Drive destination base missing or redirected')
OLD_MATRIX=DRIVE_BASE/'geant_h96_matrix3_5fe371f8aab74f42983569579477c328'; COMPLETED_BACKUPS=OLD_MATRIX/'runs'
if not OLD_MATRIX.is_dir() or OLD_MATRIX.is_symlink() or not COMPLETED_BACKUPS.is_dir() or COMPLETED_BACKUPS.is_symlink(): raise RuntimeError('Original Matrix3 backup root missing or redirected')
print('PASS: uploads pinned by SHA-256; old Drive backup root is read-only; fresh staging/Drive destinations will be generated in the preparation cell')

In [ ]:
# Every execution creates a fresh staging session; a previous partial session is never reused or cleaned.
import sys, io, os
source_bytes=resolved['geant_h96_colab_source_v1.zip']; source_manifest_bytes=resolved['source_snapshot_manifest.json']
if sha_bytes(source_bytes)!=EXPECTED['geant_h96_colab_source_v1.zip'] or sha_bytes(source_manifest_bytes)!=EXPECTED['source_snapshot_manifest.json']: raise RuntimeError('Source archive/manifest digest mismatch')
sm=json.loads(source_manifest_bytes.decode('utf-8')); entries=sm.get('entries')
if sm.get('archive_sha256')!=EXPECTED['geant_h96_colab_source_v1.zip'] or not isinstance(entries,list) or not entries: raise RuntimeError('Source manifest does not bind approved archive')
by_path={e['path']:e for e in entries}
if len(by_path)!=len(entries): raise RuntimeError('Duplicate source manifest path')
def safe_member(n):
 p=PurePosixPath(n)
 if not n or '\\' in n or p.is_absolute() or not p.parts or ':' in p.parts[0] or any(x in ('','.','..') for x in p.parts): raise RuntimeError(f'Unsafe member: {n!r}')
 return p
helper_path='phase9_geant/colab_h96_v1/notebook_stage.py'; helper_bytes=None
with zipfile.ZipFile(io.BytesIO(source_bytes)) as z:
 infos=z.infolist(); names=[i.filename for i in infos]
 if len(names)!=len({n.casefold() for n in names}) or set(names)!=(set(by_path)|{'PORTABLE_SOURCE_MANIFEST.json'}): raise RuntimeError('Source archive inventory mismatch')
 internal=json.loads(z.read('PORTABLE_SOURCE_MANIFEST.json'))
 if internal.get('schema')!='geant-h96-colab-source-snapshot-v1' or internal.get('entries')!=entries: raise RuntimeError('Internal and external source manifests differ')
 for info in infos:
  if info.is_dir() or stat.S_ISLNK((info.external_attr>>16)&0xffff): raise RuntimeError(f'Unsupported ZIP member type: {info.filename}')
  if info.filename=='PORTABLE_SOURCE_MANIFEST.json': continue
  safe_member(info.filename); payload=z.read(info)
  if len(payload)!=by_path[info.filename]['size_bytes'] or sha_bytes(payload)!=by_path[info.filename]['sha256']: raise RuntimeError(f'Source member mismatch: {info.filename}')
  if info.filename==helper_path: helper_bytes=payload
if helper_bytes is None: raise RuntimeError('Verified package omits the fresh-session staging helper')
stage_ns={'__name__':'_verified_matrix3_notebook_stage'}; exec(compile(helper_bytes,helper_path,'exec'),stage_ns)
SESSION_ID,SESSION,INCOMING=stage_ns['create_fresh_preparation_session'](Path('/content'),resolved,EXPECTED)
DRIVE_RECOVERY=DRIVE_BASE/f'geant_h96_matrix3_recovery_review18_{SESSION_ID}'
if DRIVE_RECOVERY.exists() or DRIVE_RECOVERY.is_symlink(): raise FileExistsError('Fresh Drive recovery destination exists; stop')
src_zip=INCOMING/'geant_h96_colab_source_v1.zip'; src_manifest_path=INCOMING/'source_snapshot_manifest.json'
REPO=SESSION/'approved_source_snapshot'; REPO.mkdir(exist_ok=False)
with zipfile.ZipFile(src_zip) as z:
 for info in z.infolist():
  if info.is_dir() or stat.S_ISLNK((info.external_attr>>16)&0xffff): raise RuntimeError(f'Unsupported ZIP member type: {info.filename}')
  if info.filename=='PORTABLE_SOURCE_MANIFEST.json': continue
  rel=safe_member(info.filename); payload=z.read(info)
  if len(payload)!=by_path[info.filename]['size_bytes'] or sha_bytes(payload)!=by_path[info.filename]['sha256']: raise RuntimeError(f'Source member mismatch: {info.filename}')
  target=REPO.joinpath(*rel.parts); target.parent.mkdir(parents=True,exist_ok=True)
  with target.open('xb') as f: f.write(payload); f.flush(); os.fsync(f.fileno())
print('PASS: source archive paths, exact inventory, all member hashes; fresh staging session:',SESSION)
# Only now import the hash-verified package and validate/extract train+validation files.
stage_ns['reset_prior_staged_project_imports'](sys.path,sys.modules,Path('/content'))
sys.path.insert(0,str(REPO))
from phase9_geant.colab_h96_v1 import package
data_zip=INCOMING/'geant_h96_colab_trainval_v1.zip'; data_manifest=INCOMING/'dataset_bundle_manifest.json'
package.verify_dataset_bundle(data_zip,data_manifest,EXPECTED[data_zip.name],EXPECTED[data_manifest.name])
package.extract_dataset_bundle(data_zip,REPO); DATA_ROOT=REPO/package.DATA_ROOT_RELATIVE
if {p.name for p in DATA_ROOT.iterdir()}!=set(package.DATA_ARTIFACTS): raise RuntimeError('Dataset file allowlist mismatch')
meta=package._read_dataset_metadata(DATA_ROOT)
if meta['windows']['train_count']!=3924 or meta['windows']['validation_count']!=953: raise RuntimeError('h96 dataset counts mismatch')
print('PASS: approved 9 train/validation dataset artifacts; no test arrays in bundle')

In [ ]:
TESTS=['phase9_geant/colab_h96_v1/tests/test_portability.py','phase9_geant/colab_h96_v1/tests/test_launch_matrix.py','phase9_geant/colab_h96_v1/tests/test_matrix_workflow.py','phase9_geant/colab_h96_v1/tests/test_matrix_recovery.py','phase9_geant/colab_h96_v1/tests/test_notebook_stage.py','phase9_geant/tests/test_horizon96_streaming_experiment.py','phase9_geant/tests/test_horizon96_streaming_experiment_v2.py','phase9_geant/tests/test_spatiotemporal_link.py']
test=subprocess.run([sys.executable,'-m','pytest','-q','-rs','--basetemp',str(SESSION/'pytest_synthetic'),*TESTS],cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(test.stdout)
if test.returncode!=0: raise RuntimeError(f'Synthetic suite failed (exit {test.returncode}); training remains disabled')
print('PASS: notebook-configured synthetic suite')

In [ ]:
LAUNCH=REPO/'phase9_geant/colab_h96_v1/launch_matrix_recovery.py'
ARGS=['--repository-root',str(REPO),'--source-bundle',str(src_zip),'--source-manifest',str(src_manifest_path),'--expected-source-bundle-sha256',EXPECTED[src_zip.name],'--expected-source-manifest-sha256',EXPECTED[src_manifest_path.name],'--dataset-bundle',str(data_zip),'--dataset-manifest',str(data_manifest),'--expected-dataset-bundle-sha256',EXPECTED[data_zip.name],'--expected-dataset-manifest-sha256',EXPECTED[data_manifest.name],'--completed-backup-root',str(COMPLETED_BACKUPS),'--persistent-root',str(DRIVE_RECOVERY)]
dry=subprocess.run([sys.executable,'-I','-S',str(LAUNCH),'--preflight-only',*ARGS],cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(dry.stdout)
if dry.returncode!=0: raise RuntimeError(f'Recovery preflight failed (exit {dry.returncode}); do not train')
PREFLIGHT=json.loads(dry.stdout)
if PREFLIGHT.get('status')!='preflight_passed_no_training' or len(PREFLIGHT.get('verified_existing_runs',[]))!=6: raise RuntimeError('Six completed backup verification did not pass')
if [(r['seed'],r['adjacency_mode']) for r in PREFLIGHT.get('remaining_runs',[])]!=[(123,'shuffled'),(2026,'shuffled')]: raise RuntimeError('Recovery plan includes an unintended run')
if PREFLIGHT.get('resource_policy',{}).get('policy_id')!='colab-linux-process-memory-v3-visible-cgroup': raise RuntimeError('Colab v3 memory policy mismatch')
print('PASS: six backup manifests/checkpoint/prediction hashes and scientific identities; only two remaining runs; no training')

## Explicit authorization — disabled by default

Review the separate recovery source audit and approve the exact source hashes before enabling both flags. The launcher asks a separate exact phrase before each remaining run. A completed run is copied to the fresh Drive namespace and its run manifest, checkpoint, and validation prediction hashes are read back before the next phrase is requested. If the runtime fails, preserve the partial local/Drive outputs; do not resume. A new attempt requires a separately reviewed version.

In [ ]:
SOURCE_REVIEW_APPROVED=False
RUN_TWO_REMAINING_AUTHORIZED=False
if SOURCE_REVIEW_APPROVED is not True or RUN_TWO_REMAINING_AUTHORIZED is not True: raise PermissionError('Training is disabled pending independent review and explicit authorization')
if PREFLIGHT.get('status')!='preflight_passed_no_training': raise RuntimeError('A current successful preflight is required')
if input('Type AUTHORIZE GEANT MATRIX3 RECOVERY to launch only shuffled 123 and 2026: ').strip()!='AUTHORIZE GEANT MATRIX3 RECOVERY': raise PermissionError('Recovery not authorized')
command=[sys.executable,'-I','-S',str(LAUNCH),'--run-remaining','--authorize-recovery',*ARGS]
log_path=DRIVE_BASE/f'geant_h96_matrix3_recovery_review18_{SESSION_ID}.launcher.log'
with log_path.open('x',encoding='utf-8') as log:
 proc=subprocess.Popen(command,cwd=REPO,stdin=subprocess.PIPE,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
 try:
  for line in proc.stdout:
   print(line,end='',flush=True); log.write(line); log.flush(); os.fsync(log.fileno())
   if line.startswith('AUTH_REQUIRED:'):
    phrase=input('Per-run approval (exact phrase shown above): ')
    proc.stdin.write(phrase+'\n'); proc.stdin.flush()
  exit_code=proc.wait()
 except BaseException:
  proc.kill(); proc.wait(); raise
if exit_code!=0: raise RuntimeError(f'Recovery stopped with exit {exit_code}; preserve partial artifacts; never retry this namespace')
summary_path=DRIVE_RECOVERY/'combined_results.json'; summary_bytes=summary_path.read_bytes(); summary=json.loads(summary_bytes)
complete_path=DRIVE_RECOVERY/'matrix_lifecycle'/'matrix_run_003_complete.json'; complete=json.loads(complete_path.read_text(encoding='utf-8'))
if sha_bytes(summary_bytes)!=complete.get('combined_results_sha256'): raise RuntimeError('Combined result summary hash mismatch')
expected_all={(123,'topology'),(2026,'topology'),(42,'identity'),(123,'identity'),(2026,'identity'),(42,'shuffled'),(123,'shuffled'),(2026,'shuffled')}
if summary.get('status')!='complete' or {(r['seed'],r['adjacency_mode']) for r in summary.get('runs',[])}!=expected_all or summary.get('test_access') is not False: raise RuntimeError('Combined result set/status/test-seal mismatch')
new_runs=DRIVE_RECOVERY/'runs'
if {p.name for p in new_runs.iterdir()}!={'h96_shuffled_seed_123','h96_shuffled_seed_2026'}: raise RuntimeError('New backup inventory mismatch')
for seed in (123,2026):
 d=new_runs/f'h96_shuffled_seed_{seed}'; bm=json.loads((d/'backup_manifest.json').read_text()); rm=json.loads((d/'run.json').read_text())
 if {p.name for p in d.iterdir()}!={'run.json','best.pt','validation_predictions.npy','backup_manifest.json'}: raise RuntimeError('Unexpected run artifact')
 if bm.get('status')!='complete' or bm.get('readback_verified') is not True or rm.get('status')!='complete' or rm.get('seed')!=seed or rm.get('adjacency_mode')!='shuffled' or rm.get('test_access',{}).get('permitted') is not False: raise RuntimeError('New run manifest identity mismatch')
 for name in ('run.json','best.pt','validation_predictions.npy'):
  if sha_bytes((d/name).read_bytes())!=bm['artifacts'][name]['sha256']: raise RuntimeError(f'Backup hash mismatch: {d.name}/{name}')
 if bm.get('source_run_manifest_sha256')!=sha_bytes((d/'run.json').read_bytes()): raise RuntimeError('Backup does not bind its run manifest')
 if rm.get('checkpoint_sha256')!=bm['artifacts']['best.pt']['sha256'] or rm.get('validation_predictions_sha256')!=bm['artifacts']['validation_predictions.npy']['sha256']: raise RuntimeError('Run and backup artifact digests disagree')
print('RECOVERY COMPLETE: integrity-checked eight-run summary',summary_path,'launcher log',log_path)